In [1]:
import pandas as pd

import numpy as np

import pickle

import mlflow.sklearn

import mlflow.tracking

from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import logging

import warnings
warnings.filterwarnings('ignore')



In [2]:
import os

os.getcwd()

'C:\\Users\\sarav\\Smart_Premium\\Smart_premium_ML'

In [3]:
test_file = "C:\\Users\\sarav\\Smart_Premium\\Smart_premium_ML\\test.csv"

test_data = pd.read_csv(test_file)

test_data

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type
0,1200000,28.0,Female,2310.0,NaN,4.0,Bachelor's,Self-Employed,7.657981,Rural,Basic,NaN,19.0,NaN,1.0,2023-06-04 15:21:39.245086,Poor,Yes,Weekly,House
1,1200001,31.0,Female,126031.0,Married,2.0,Master's,Self-Employed,13.381379,Suburban,Premium,NaN,14.0,372.0,8.0,2024-04-22 15:21:39.224915,Good,Yes,Rarely,Apartment
2,1200002,47.0,Female,17092.0,Divorced,0.0,PhD,Unemployed,24.354527,Urban,Comprehensive,NaN,16.0,819.0,9.0,2023-04-05 15:21:39.134960,Average,Yes,Monthly,Condo
3,1200003,28.0,Female,30424.0,Divorced,3.0,PhD,Self-Employed,5.136225,Suburban,Comprehensive,1.0,3.0,770.0,5.0,2023-10-25 15:21:39.134960,Poor,Yes,Daily,House
4,1200004,24.0,Male,10863.0,Divorced,2.0,High School,Unemployed,11.844155,Suburban,Premium,NaN,14.0,755.0,7.0,2021-11-26 15:21:39.259788,Average,No,Weekly,House
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799995,1999995,50.0,Female,38782.0,Married,1.0,Bachelor's,NaN,14.498639,Rural,Premium,NaN,8.0,309.0,2.0,2021-07-09 15:21:39.184157,Average,Yes,Daily,Condo
799996,1999996,NaN,Female,73462.0,Single,0.0,Master's,NaN,8.145748,Rural,Basic,2.0,0.0,NaN,2.0,2023-03-28 15:21:39.250151,Good,No,Daily,Apartment
799997,1999997,26.0,Female,35178.0,Single,0.0,Master's,Employed,6.636583,Urban,Comprehensive,NaN,10.0,NaN,6.0,2019-09-30 15:21:39.132191,Poor,No,Monthly,Apartment
799998,1999998,34.0,Female,45661.0,Single,3.0,Master's,NaN,15.937248,Urban,Premium,2.0,17.0,467.0,7.0,2022-05-09 15:21:39.253660,Average,No,Weekly,Condo


In [4]:
test_data.drop(['id','Policy Start Date'], axis=1, inplace=True)

In [5]:
for col in test_data.select_dtypes(include=['int64', 'float64']).columns:
    test_data[col].fillna(test_data[col].mean(), inplace=True)

for col in test_data.select_dtypes(include='object').columns:
    test_data[col].fillna(test_data[col].mode()[0])

In [6]:
def age_category(data):
    if 18 < data <= 30: return '18-30'
    elif 30 < data <= 40: return '31-40'
    elif 40 < data <= 50: return '41-50'
    elif 50 < data <= 64: return '51-64'
    else: return '<64'

In [7]:
def dependent_category(data):
    if data == 0: return '0'
    elif 0 < data <= 2: return '0-2'
    elif 2 < data <= 3: return '2-3'
    else: return '<3'

In [8]:
def health_category(data):
    if 0 < data <= 15: return '0-15'
    elif 15 < data <= 25: return '15-25'
    elif 25 < data <= 35: return '15-35'
    else: return '<35'

In [9]:
def claims(data):
    if 0 < data <= 1: return '0-1'
    elif 1 < data <= 2: return '1-2'
    else: return '<2'

In [10]:
def vehicle(data):
    if 0 < data <= 5: return '0-5'
    elif 5 < data <= 10: return '5-10'
    elif 10 < data <= 20: return '10-20'
    else: return '<20'

In [11]:
def credit(data):
    if 0 < data <= 300: return '0-300'
    elif 300 < data <= 600: return '300-600'
    elif 600 < data < 800: return '600-800'
    else: return '<800'

In [12]:
def insurance(data):
    if 0 < data <= 3: return '0-3'
    elif 3 < data <= 6: return '3-6'
    elif 6 < data < 9: return '6-9'
    else: return '<9'

In [13]:
test_data['Age_Group'] = test_data['Age'].apply(age_category)
test_data['Dependent_Group'] = test_data['Number of Dependents'].apply(dependent_category)
test_data['Health_Group'] = test_data['Health Score'].apply(health_category)
test_data['Prev_Claims_Group'] = test_data['Previous Claims'].apply(claims)
test_data['Vehicle_Group'] = test_data['Vehicle Age'].apply(vehicle)
test_data['Credit_Group'] = test_data['Credit Score'].apply(credit)
test_data['Insurance_Group'] = test_data['Insurance Duration'].apply(insurance)

In [14]:
mappings = {
    "Education Level": {"High School": 0, "Bachelor's": 1, "Master's": 2, "PhD": 3},
    "Customer Feedback": {"Poor": 0, "Average": 1, "Good": 2},
    "Exercise Frequency": {"Rarely": 0, "Weekly": 1, "Monthly": 2, "Daily": 3},
    "Policy Type": {"Basic": 0, "Comprehensive": 1, "Premium": 2}
}

In [15]:
test_data = test_data.replace(mappings).infer_objects(copy=False)

In [16]:
columns_to_encode = test_data[['Age_Group', 'Dependent_Group', 'Health_Group', 'Prev_Claims_Group',
                               'Vehicle_Group', 'Credit_Group', 'Insurance_Group', 'Gender', 'Marital Status',
                               'Occupation', 'Location', 'Smoking Status', 'Property Type']]

In [17]:
le = LabelEncoder()
for col in columns_to_encode.columns:
    test_data[col] = le.fit_transform(test_data[col])

In [18]:
encoded_test_data = pd.DataFrame({
    'Age': test_data['Age_Group'],
    'Gender': test_data['Gender'],
    'Annual Income': test_data['Annual Income'],
    'Marital Status': test_data['Marital Status'],
    'Number of Dependents': test_data['Dependent_Group'],
    'Education Level': test_data['Education Level'],
    'Occupation': test_data['Occupation'],
    'Health Score': test_data['Health_Group'],
    'Location': test_data['Location'],
    'Policy Type': test_data['Policy Type'],
    'Previous Claims': test_data['Prev_Claims_Group'],
    'Vehicle Age': test_data['Vehicle_Group'],
    'Credit Score': test_data['Credit_Group'],
    'Insurance Duration': test_data['Insurance_Group'],
    'Customer Feedback': test_data['Customer Feedback'],
    'Smoking Status': test_data['Smoking Status'],
    'Exercise Frequency': test_data['Exercise Frequency'],
    'Property Type': test_data['Property Type']
})

In [19]:
def log_transform(data, columns_to_transform):
    for col in columns_to_transform:
        data[f'{col}_log'] = np.log1p(data[col])  
        data[f'{col}_msle'] = (data[f'{col}_log'] - data[f'{col}_log'].mean())**2  
        data[col] = np.sqrt(data[f'{col}_msle'])  
        data.drop(columns=[f'{col}_log', f'{col}_msle'], inplace=True)  
    
    return data

In [20]:
log_transform(encoded_test_data, ['Annual Income'])

,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Customer Feedback,Smoking Status,Exercise Frequency,Property Type
0,0,0,2.047339,3,3,1,1,0,0,0,1,1,1,0,0.0,1,1,2
1,1,0,1.951517,1,1,2,1,0,1,2,1,1,1,2,2.0,1,0,0
2,2,0,0.046350,0,0,3,2,1,2,1,1,1,3,3,1.0,1,2,1
3,0,0,0.530245,0,2,3,1,0,1,1,0,0,2,1,0.0,1,3,2
4,0,1,0.499565,0,1,0,2,0,1,2,1,1,2,2,1.0,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799995,2,0,0.772963,1,1,1,3,0,0,2,1,2,1,0,1.0,1,3,1
799996,2,0,1.411763,2,0,2,3,0,0,0,1,3,1,0,2.0,0,3,0
799997,0,0,0.675430,2,0,2,0,0,2,1,1,2,1,1,0.0,0,2,0
799998,1,0,0.936247,2,2,2,3,1,2,2,1,1,1,2,1.0,0,1,1


In [25]:
with open("best_model.pkl", "rb") as file:
    model = pickle.load(file)

model

DecisionTreeRegressor(max_depth=19,
                      min_samples_split=np.float64(2.8132102948803457))